# Google Compute Engine 실시간 최저가 리전 탐색 및 프로비저닝

이 노트북은 **Google Cloud Billing Catalog API**를 실시간으로 직접 호출하여 전 세계 40개 이상의 모든 GCP 리전에서 `e2-medium` (1.0 Core + 4.0 GB RAM) 및 10GB `pd-balanced` 영구 디스크의 최신 단가를 수집·비교하고, **가장 저렴한 리전 Top 3를 실시간으로 출력한 뒤 해당 리전에 VM을 생성**하는 자동화 예제입니다.

### 대상 인스턴스 사양
- **프로젝트**: `iceu-songpa09`
- **실행 계정**: `songpa09@iceu.kr`
- **인스턴스명**: `instance-20260914-063020`
- **머신 유형**: `e2-medium` (2 vCPU, 4GB Memory)
- **OS 이미지**: Debian 13 (Trixie, 무료 OS 라이선스)
- **디스크**: 10GB `pd-balanced` (자동 삭제: Yes)
- **프로비저닝 모델**: STANDARD (On-Demand)

## 1. 사전 인증 및 계정/프로젝트 설정
`iceu-songpa09` 프로젝트의 리소스 생성 권한을 가진 `songpa09@iceu.kr` 계정으로 설정합니다.

In [ ]:
!gcloud config set account songpa09@iceu.kr
!gcloud config set project iceu-songpa09
!gcloud auth list

## 2. [실시간 API] Cloud Billing Catalog API 호출 및 전 세계 리전 최저가 Top 3 분석

Google Cloud의 공식 Billing Catalog API (`/v1/services/6F81-5844-456A/skus`)를 실시간으로 호출하여 다음 단가를 집계합니다:
1. 전 세계 리전별 `E2 Instance Core` (시간당 코어 단가)
2. 전 세계 리전별 `E2 Instance Ram` (GB-시간당 메모리 단가)
3. 전 세계 리전별 `Balanced PD Capacity` (GB-월당 스토리지 단가)

수집된 실시간 단가를 기반으로 **`e2-medium` + 10GB `pd-balanced` 월간 총비용(730시간 기준)**을 계산하여 가장 저렴한 리전 Top 3를 출력합니다.

In [ ]:
import subprocess
import urllib.request
import json
import time
import sys
import os

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

PROJECT = "iceu-songpa09"
ACCOUNT = "songpa09@iceu.kr"
GCLOUD_CMD = "gcloud.cmd" if os.name == "nt" else "gcloud"

print("[1/4] Compute Engine 가용 리전 및 영역(Zone) 실시간 수집 중...")
start_time = time.time()

# 1. 실제 인스턴스 생성이 가능한 Compute Engine Zone 및 Region 매핑 조회 (가상/미출시 리전 자동 배제)
zone_cmd = f"{GCLOUD_CMD} compute zones list --format=\"csv[no-heading](name,region)\" --project={PROJECT} --account={ACCOUNT}"
zones_raw = subprocess.check_output(zone_cmd, shell=True, text=True)

zone_map = {}
for line in zones_raw.strip().splitlines():
    if ',' in line:
        z, r = line.strip().split(',')
        zone_map.setdefault(r, []).append(z)

# 2. gcloud 인증 토큰 획득
print("[2/4] Cloud Billing API 인증 토큰 획득 중...")
token = subprocess.check_output(f"{GCLOUD_CMD} auth print-access-token --account={ACCOUNT}", shell=True, text=True).strip()

page_token = ''
region_cores = {}
region_rams = {}
region_disks = {}
page_count = 0

# 3. Compute Engine 서비스 SKU 카탈로그 페이지네이션 조회 (Service ID: 6F81-5844-456A)
print("[3/4] Google Cloud Billing Catalog API SKU 데이터 수집 중...")
while True:
    url = 'https://cloudbilling.googleapis.com/v1/services/6F81-5844-456A/skus?pageSize=5000'
    if page_token:
        url += f'&pageToken={page_token}'
    req = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}'})
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode('utf-8'))
        page_count += 1
        for sku in data.get('skus', []):
            desc = sku.get('description', '')
            regions = sku.get('serviceRegions', [])
            if not regions:
                continue
            region = regions[0]
            
            # E2 표준 코어 단가 (On-Demand)
            if desc.startswith('E2 Instance Core running in') and 'Preemptible' not in desc and 'Spot' not in desc:
                p = sku['pricingInfo'][0]['pricingExpression']['tieredRates'][0]['unitPrice']
                rate = float(p.get('units', 0)) + float(p.get('nanos', 0)) / 1e9
                region_cores[region] = rate
                
            # E2 표준 RAM 단가 (On-Demand)
            elif desc.startswith('E2 Instance Ram running in') and 'Preemptible' not in desc and 'Spot' not in desc:
                p = sku['pricingInfo'][0]['pricingExpression']['tieredRates'][0]['unitPrice']
                rate = float(p.get('units', 0)) + float(p.get('nanos', 0)) / 1e9
                region_rams[region] = rate
                
            # Balanced PD 디스크 단가
            elif desc.startswith('Balanced PD Capacity') and 'Regional' not in desc:
                p = sku['pricingInfo'][0]['pricingExpression']['tieredRates'][0]['unitPrice']
                rate = float(p.get('units', 0)) + float(p.get('nanos', 0)) / 1e9
                region_disks[region] = rate
                
        page_token = data.get('nextPageToken')
        if not page_token:
            break

# 4. 실제 생성 가능한 가용 리전만 필터링하여 월간 비용(730시간) 계산
print("[4/4] 실제 가용 리전 비용 비교 및 최적 리전/영역 산출 중...")
results = []
valid_regions = set(region_cores.keys()) & set(region_rams.keys()) & set(zone_map.keys())

for r in valid_regions:
    core_price = region_cores[r]
    ram_price = region_rams[r]
    disk_price = region_disks.get(r, 0.10)
    
    vm_hourly = (1.0 * core_price) + (4.0 * ram_price)
    vm_monthly = vm_hourly * 730
    disk_monthly = disk_price * 10
    total_monthly = vm_monthly + disk_monthly
    
    results.append({
        'region': r,
        'primary_zone': zone_map[r][0],
        'core_hourly': core_price,
        'ram_hourly_gb': ram_price,
        'vm_hourly': vm_hourly,
        'vm_monthly': vm_monthly,
        'disk_monthly': disk_monthly,
        'total_monthly': total_monthly
    })

# 비용 기준 오름차순 정렬
results.sort(key=lambda x: x['total_monthly'])

print(f"\n[조회 완료] 총 {len(results)}개 실제 가용 리전 분석 완료 (소요 시간: {time.time()-start_time:.1f}초)\n")

print("=" * 90)
print("[실시간 Cloud Billing API 결과] 실제 가용 리전 중 가장 저렴한 리전 TOP 3")
print("=" * 90)
print(f"{'순위':<4} | {'리전 코드':<18} | {'추천 영역(Zone)':<18} | {'VM 시간당':<12} | {'총 월 예상 비용'}")
print("-" * 90)
for idx, item in enumerate(results[:3], 1):
    print(f"#{idx:<3} | {item['region']:<18} | {item['primary_zone']:<18} | ${item['vm_hourly']:.4f}/hr    | ${item['total_monthly']:.2f}/월")
print("=" * 90)

# 전역 변수 저장
cheapest_region = results[0]['region']
cheapest_zone = results[0]['primary_zone']
print(f"\n[자동 지정] 최저가 1위 리전 [{cheapest_region}] / 유효 영역 [{cheapest_zone}]으로 설정되었습니다.")

## 3. Ops Agent 정책 설정 파일 (`config.yaml`) 생성
인스턴스 라벨(`goog-ops-agent-policy: v2-template-1-7-0`)과 일치하는 VM에 Ops Agent 최신 버전을 설치하도록 규칙을 정의합니다.

In [ ]:
%%writefile config.yaml
agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0

## 4. [동적 배포] 최저가 리전으로 Compute Engine VM 생성 및 Ops Agent 배포
위 2번 셀에서 **실시간 API로 판별된 최저가 리전(`cheapest_region`)**의 영역(Zone)을 자동으로 지정하여 VM을 생성하고 Ops Agent 정책을 배포합니다.

- 인스턴스 중복 및 Ops Agent 정책 중복(`ALREADY_EXISTS`) 자동 감지 및 갱신(Update) 처리
- 윈도우 인코딩(CP949) 및 실행 파일(`gcloud.cmd`) 안정화 적용

In [ ]:
import subprocess
import os
import sys

# 윈도우 CP949 인코딩 충돌 방지
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# -----------------------------------------------------
# [기본 설정]
# -----------------------------------------------------
PROJECT = "iceu-songpa09"
ACCOUNT = "songpa09@iceu.kr"   # 권한이 있는 계정 명시
INSTANCE_NAME = "instance-20260915-143300"
GCLOUD_CMD = "gcloud.cmd" if os.name == "nt" else "gcloud"

# 2번 셀에서 계산된 최저가 리전 및 실제 존재하는 영역(Zone) 자동 바인딩
TARGET_REGION = globals().get("cheapest_region", "us-central1")
TARGET_ZONE = globals().get("cheapest_zone", f"{TARGET_REGION}-a")
POLICY_NAME = f"goog-ops-agent-v2-template-1-7-0-{TARGET_ZONE}"

print(f"[배포 준비] 프로젝트: {PROJECT} | 실행 계정: {ACCOUNT}")
print(f"[타깃 리전/영역]: {TARGET_REGION} / {TARGET_ZONE}")

# config.yaml 파일이 없으면 자동 생성
if not os.path.exists("config.yaml"):
    print("[안내] config.yaml 파일이 없어 자동으로 생성합니다...")
    with open("config.yaml", "w", encoding="utf-8") as f:
        f.write("""agentsRule:
  packageState: installed
  version: latest
instanceFilter:
  inclusionLabels:
  - labels:
      goog-ops-agent-policy: v2-template-1-7-0
""")
    print("[OK] config.yaml 생성 완료.")

# disk-resource-policy 설정 (us-central1 리전에만 존재하는 경우 처리)
disk_policy_arg = ""
if TARGET_REGION == "us-central1":
    disk_policy_arg = f",disk-resource-policy=projects/{PROJECT}/regions/{TARGET_REGION}/resourcePolicies/default-schedule-1"

# 1. Compute VM 인스턴스 존재 여부 사전 확인 (중복 에러 방지)
check_vm = subprocess.run(
    f"{GCLOUD_CMD} compute instances describe {INSTANCE_NAME} --zone={TARGET_ZONE} --project={PROJECT} --account={ACCOUNT} --format=\"value(status)\"",
    shell=True, capture_output=True, text=True
)

if check_vm.returncode == 0:
    print(f"[안내] 인스턴스({INSTANCE_NAME})가 이미 존재합니다 (상태: {check_vm.stdout.strip()}).")
else:
    print("\n[1/2] Compute Engine VM 인스턴스 생성 중...")
    create_cmd = (
        f"{GCLOUD_CMD} compute instances create {INSTANCE_NAME} "
        f"--project={PROJECT} "
        f"--account={ACCOUNT} "
        f"--zone={TARGET_ZONE} "
        "--machine-type=e2-medium "
        "--network-interface=network-tier=PREMIUM,stack-type=IPV4_ONLY,subnet=default "
        "--metadata=enable-osconfig=TRUE "
        "--maintenance-policy=MIGRATE "
        "--provisioning-model=STANDARD "
        "--service-account=108335720396-compute@developer.gserviceaccount.com "
        "--scopes=https://www.googleapis.com/auth/devstorage.read_only,https://www.googleapis.com/auth/logging.write,https://www.googleapis.com/auth/monitoring.write,https://www.googleapis.com/auth/service.management.readonly,https://www.googleapis.com/auth/servicecontrol,https://www.googleapis.com/auth/trace.append "
        f"--create-disk=auto-delete=yes,boot=yes,device-name={INSTANCE_NAME}{disk_policy_arg},image=projects/debian-cloud/global/images/debian-13-trixie-v20260908,mode=rw,size=10,type=pd-balanced "
        "--no-shielded-secure-boot "
        "--shielded-vtpm "
        "--shielded-integrity-monitoring "
        "--labels=goog-ops-agent-policy=v2-template-1-7-0,goog-ec-src=vm_add-gcloud "
        "--reservation-affinity=any"
    )
    res1 = subprocess.run(create_cmd, shell=True, capture_output=True, text=True, encoding='utf-8', errors='replace')
    if res1.returncode == 0:
        print("[OK] [1/2] VM 인스턴스 생성 완료!")
        if res1.stdout:
            print(res1.stdout.strip())
    else:
        print("[오류 발생] VM 인스턴스 생성 실패:")
        print(res1.stderr if res1.stderr else res1.stdout)

# 2. Ops Agent 정책 존재 여부 확인 후 적용 (이미 있으면 update, 없으면 create)
print("\n[2/2] Ops Agent 정책 적용 중...")
check_policy = subprocess.run(
    f"{GCLOUD_CMD} compute instances ops-agents policies describe {POLICY_NAME} --zone={TARGET_ZONE} --project={PROJECT} --account={ACCOUNT}",
    shell=True, capture_output=True, text=True
)

if check_policy.returncode == 0:
    # 정책이 이미 존재하는 경우 갱신(update) 실행
    policy_cmd = (
        f"{GCLOUD_CMD} compute instances ops-agents policies update {POLICY_NAME} "
        f"--project={PROJECT} "
        f"--account={ACCOUNT} "
        f"--zone={TARGET_ZONE} "
        "--file=config.yaml"
    )
    res2 = subprocess.run(policy_cmd, shell=True, capture_output=True, text=True, encoding='utf-8', errors='replace')
    print(f"[성공] 기존 Ops Agent 정책({POLICY_NAME})을 최신 설정으로 갱신하여 적용했습니다!")
else:
    # 정책이 없는 경우 신규 생성(create) 실행
    policy_cmd = (
        f"{GCLOUD_CMD} compute instances ops-agents policies create {POLICY_NAME} "
        f"--project={PROJECT} "
        f"--account={ACCOUNT} "
        f"--zone={TARGET_ZONE} "
        "--file=config.yaml"
    )
    res2 = subprocess.run(policy_cmd, shell=True, capture_output=True, text=True, encoding='utf-8', errors='replace')
    if res2.returncode == 0:
        print(f"[성공] 새 Ops Agent 정책({POLICY_NAME})이 배포되었습니다!")
    else:
        print(f"[오류] Ops Agent 정책 배포 실패:\n{res2.stderr}")

## 5. 생성 결과 확인
VM 인스턴스와 Ops Agent 정책이 정상 배포되었는지 확인합니다.

In [ ]:
!gcloud compute instances list --project=iceu-songpa09 --account=songpa09@iceu.kr
!gcloud compute instances ops-agents policies list --project=iceu-songpa09 --account=songpa09@iceu.kr